# TRIBE v2 — Backend on Colab
Runs the FastAPI backend + model on Colab's GPU and exposes it via a public URL.

**Steps:**
1. Run all cells top to bottom
2. Copy the public URL printed in the last cell
3. Paste it into the Backend field on your Vercel frontend

In [ ]:
# Check GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Install system dependencies
!apt-get install -y -q ffmpeg

In [ ]:
# Install Python dependencies
!pip install -q fastapi uvicorn yt-dlp numpy pydantic "uvicorn[standard]" nest_asyncio huggingface_hub whisperx

In [ ]:
# Clone and install tribev2
!git clone -q https://github.com/facebookresearch/tribev2
!pip install -q -e tribev2

In [ ]:
# Clone the brain-neuro repo
!git clone -q https://github.com/Sambhram1/brain-neuro.git
%cd brain-neuro

In [ ]:
# Download model weights from HuggingFace
# Get your token at https://huggingface.co/settings/tokens
# Add it as a Colab secret named HF_TOKEN (key icon in left sidebar)
from google.colab import userdata
import huggingface_hub

try:
    hf_token = userdata.get('HF_TOKEN')
    print('Using HF_TOKEN from Colab secrets')
except Exception:
    hf_token = input('Enter your HuggingFace token: ')

huggingface_hub.login(token=hf_token, add_to_git_credential=False)
huggingface_hub.hf_hub_download(
    repo_id='facebook/tribev2',
    filename='best.ckpt',
    local_dir='model_weights'
)
print('Model weights downloaded.')

In [ ]:
# Set device to cuda in config (if GPU available)
import re

with open('model_weights/config.yaml') as f:
    cfg = f.read()

target_device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg = re.sub(r'device: (cpu|cuda)', f'device: {target_device}', cfg)

with open('model_weights/config.yaml', 'w') as f:
    f.write(cfg)

print(f'Config patched: all extractors → device: {target_device}')

In [ ]:
# Install cloudflared (free tunnel, no account needed)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('cloudflared ready')

In [ ]:
# Start backend + tunnel
import nest_asyncio, uvicorn, threading, subprocess, time, re as _re, sys

nest_asyncio.apply()

# Start FastAPI in a background thread
def run_server():
    uvicorn.run('backend:app', host='0.0.0.0', port=8000, log_level='warning')

t = threading.Thread(target=run_server, daemon=True)
t.start()
time.sleep(3)
print('FastAPI started on port 8000')

# Start cloudflared tunnel and capture the public URL
proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

print('Waiting for tunnel URL...')
for line in proc.stdout:
    line = line.decode()
    match = _re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group()
        print(f'\n=========================================')
        print(f'PUBLIC BACKEND URL:')
        print(f'  {public_url}')
        print(f'=========================================')
        print(f'Paste this URL into the Backend field on your Vercel frontend.')
        break